In [1]:
%load_ext autoreload
%autoreload 2

import pathlib, os

# Project root, whether the kernel cwd is the repo root or literatureComparison/
ROOT = pathlib.Path.cwd()
if not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

import numpy as np
from astropy.table import Table
from astropy import units as u
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display, HTML
from crossmatching import Crossmatcher, EMCCatalog, EMCIdSupplier, NEACatalog, SimbadIdSupplier, ParamFiller, temperate_mask, rocky_mask
from crossmatching.enrichment import (
    HpicParamSource, NeaParamSource, SimbadParamSource,
    EpicParamSource, ToiParamSource, EuParamSource
)
from collections import Counter
import glob
import ipydatagrid
import matplotlib

mpl.rcParams['figure.dpi'] = 300

In [ ]:
nea_src = NeaParamSource()
nea_src.load(from_file='././input/pscomppars.txt', format='ascii')
print(f'NEA planets loaded:     {len(nea_src._lookup):,}')
print(f'With insol:             {sum(1 for v in nea_src._lookup.values() if "insol" in v):,}')
print(f'With teff + rad:        {sum(1 for v in nea_src._lookup.values() if "teff" in v and "rad" in v):,}')


# Tier 3
eu_src = EuParamSource()
eu_path = sorted(glob.glob('../Exo-MerCat/InputSources/eu_init*.csv'))[-1]
eu_src.load(from_file=eu_path, format="ascii.csv")
print(f"\nEU planets loaded: {len(eu_src._lookup):,}")

# ── Tier 2: EPIC (K2 planets not in NEA) ─────────────────────────────────────
epic_src = EpicParamSource()
epic_path = sorted(glob.glob('../Exo-MerCat/InputSources/epic_init*.csv'))[-1]
epic_src.load(from_file=epic_path, format='ascii.csv')
print(f'\nEPIC planets loaded:    {len(epic_src._lookup):,}')
print(f'With insol:             {sum(1 for v in epic_src._lookup.values() if "insol" in v):,}')
print(f'With teff + rad:        {sum(1 for v in epic_src._lookup.values() if "teff" in v and "rad" in v):,}')

# ── Tier 3: TOI (TESS objects) ────────────────────────────────────────────────
toi_src = ToiParamSource()
toi_path = sorted(glob.glob('../Exo-MerCat/InputSources/toi_init*.csv'))[-1]
toi_src.load(from_file=toi_path, format='ascii.csv')
print(f'\nTOI entries loaded:     {len(toi_src._lookup):,}')
print(f'With insol:             {sum(1 for v in toi_src._lookup.values() if "insol" in v):,}')
print(f'With teff + rad:        {sum(1 for v in toi_src._lookup.values() if "teff" in v and "rad" in v):,}')

# ── Tier 4: SIMBAD fallback ───────────────────────────────────────────────────
simbad_src = SimbadParamSource()
simbad_src.load(from_file='./input/simbad_params.txt')
print(f'SIMBAD matches: {len(simbad_src._lookup):,}')

merger = ParamFiller([nea_src, eu_src, toi_src, toi_src, simbad_src])

NEA planets loaded:     6,298
With insol:             4,413
With teff + rad:        5,977

EU planets loaded: 11,159

EPIC planets loaded:    1,806
With insol:             175
With teff + rad:        1,230

TOI entries loaded:     7,931
With insol:             7,752
With teff + rad:        7,394
SIMBAD matches: 8,930


In [3]:
hpic = Table.read("./input/HPIC_LC4_combined_d50.txt", format="ascii")

In [ ]:
cme = Crossmatcher(EMCCatalog(), EMCIdSupplier())
cme.load_catalog(from_file="./input/exo-mercat.csv", format='csv')
cme.load_alternate_ids(hpic['star_name'], from_file='./input/exo-mercat.csv')

cm = Crossmatcher(NEACatalog(), SimbadIdSupplier())
cm.load_catalog(from_file="././input/pscomppars.txt", format="ascii")
cm.load_alternate_ids(hpic['star_name'], from_file='./input/alternate_ids_hpic.txt')

hpic = cme.remove_duplicates(hpic, input_starname_key="star_name")
cm_nea = cm.combined_crossmatch(hpic, input_starname_key="star_name", input_epoch=2000)
cm_emc = cme.combined_crossmatch(hpic, input_starname_key="star_name", input_epoch=2000)
full_nea = cm.catalog_table.copy()
full_emc = cme.catalog_table.copy()
print(len(hpic))

14559


In [5]:
e_cm_emc = merger.enrich(
    cm_emc,
    **EMCCatalog.ENRICH_KEYS,
    id_supplier=cme.id_supplier,
    alternate_ids=cme.alternate_ids
)[0]
e_full_emc = merger.enrich(
    full_emc,
    **EMCCatalog.ENRICH_KEYS,
    id_supplier=cme.id_supplier,
    alternate_ids=cme.alternate_ids
)[0]
e_cm_nea = merger.enrich(
    cm_nea,
    **NEACatalog.ENRICH_KEYS,
    id_supplier=cm.id_supplier,
    alternate_ids=cm.alternate_ids
)[0]
e_full_nea = merger.enrich(
    full_nea,
    **NEACatalog.ENRICH_KEYS,
    id_supplier=cm.id_supplier,
    alternate_ids=cm.alternate_ids
)[0]

print(f"enriched full EMC:        {len(e_full_emc):7d}")
print(f"enriched crossmatched EMC:{len(e_cm_emc):7d}")
print(f"enriched full NEA:        {len(e_full_nea):7d}")
print(f"enriched crossmatched NEA:{len(e_cm_nea):7d}")


enriched full EMC:          16097
enriched crossmatched EMC:   1260
enriched full NEA:           6298
enriched crossmatched NEA:    849


In [6]:
e_full_emc.colnames

['exo-mercat_name',
 'nasa_name',
 'toi_name',
 'epic_name',
 'eu_name',
 'oec_name',
 'host',
 'letter',
 'main_id',
 'binary',
 'main_id_ra',
 'main_id_dec',
 'mass',
 'mass_max',
 'mass_min',
 'mass_url',
 'msini',
 'msini_max',
 'msini_min',
 'msini_url',
 'bestmass',
 'bestmass_max',
 'bestmass_min',
 'bestmass_url',
 'bestmass_provenance',
 'p',
 'p_max',
 'p_min',
 'p_url',
 'r',
 'r_max',
 'r_min',
 'r_url',
 'a',
 'a_max',
 'a_min',
 'a_url',
 'e',
 'e_max',
 'e_min',
 'e_url',
 'i',
 'i_max',
 'i_min',
 'i_url',
 'discovery_method',
 'status',
 'checked_status_string',
 'original_status_string',
 'confirmed',
 'discovery_year',
 'main_id_aliases',
 'main_id_provenance',
 'angular_separation_flag',
 'angular_separation',
 'catalog',
 'duplicate_catalog_flag',
 'duplicate_names',
 'binary_coordinate_mismatch_flag',
 'binary_complex_system_flag',
 'coordinate_mismatch_flag',
 'coordinate_mismatch',
 'period_mismatch_flag',
 'fallback_merge_flag',
 'misnamed_duplicates_flag',
 'r

In [7]:
for t in (e_cm_emc, e_full_emc ):
    t['r_earth']     = t['r'] * u.R_jup.to(u.R_earth)
    t['mass_earth']  = t['mass'] * u.M_jup.to(u.M_earth)
    t['msini_earth'] = t['msini'] * u.M_jup.to(u.M_earth)
    t['r_earth_lower_bound'] = t['r_lower_bound'] * u.R_jup.to(u.R_earth)
    t['r_earth_upper_bound'] = t['r_upper_bound'] * u.R_jup.to(u.R_earth)

for t in (e_cm_nea, e_full_nea ):
    t['r_earth']     = t['pl_rade']
    t['mass_earth']  = t['pl_masse']
    t['msini_earth'] = t['pl_msinie']
    t['r_earth_lower_bound'] = t['pl_radj_lower_bound'] * u.R_jup.to(u.R_earth)
    t['r_earth_upper_bound'] = t['pl_radj_upper_bound'] * u.R_jup.to(u.R_earth)


In [8]:
((full_emc["r"].mask) & ~(full_emc["mass"].mask)).sum()

np.int64(1825)

In [9]:
Counter(full_emc["bestmass_provenance"].filled(''))

Counter({np.str_(''): 11055, np.str_('Mass'): 3828, np.str_('Msini'): 1214})

In [13]:
CATEGORY_ORDER  = ['Sun-like', 'Low-luminosity', 'Very-low-luminosity', 'Other']
CATEGORY_COLORS = {
    'Sun-like':            '#F5A623',
    'Low-luminosity':      '#E05C00',
    'Very-low-luminosity': '#C0392B',
    'Other':               '#95A5A6',
}

# Kopparapu et al. 2013 (erratum 2014) -- optimistic bounds
KOPP_RECENT_VENUS = (1.776, 2.136e-4, 2.533e-8, -1.332e-11, -3.097e-15)
KOPP_EARLY_MARS   = (0.320, 5.547e-5, 1.526e-9, -2.874e-12, -5.011e-16)

def kopparapu_seff(teff, coeffs):
    s0, a, b, c, d = coeffs
    ts = np.clip(np.asarray(teff, dtype=float), 2600.0, 7200.0) - 5780.0
    return s0 + a*ts + b*ts**2 + c*ts**3 + d*ts**4

import re
_BAYER_PREFIX = re.compile(r'^(alf|bet|gam|del|eps|zet|eta|tet|iot|kap|lam|mu|nu|xi|omi|pi|rho|sig|tau|ups|phi|chi|psi|ome)(0\d)\b', re.IGNORECASE)

def normalize_name(s):
    """Cross-catalog name match key. Strips SIMBAD '*' prefix and 2-digit suffix
    on lowercase Bayer letters ('alf01 Cen' -> 'alf Cen'). Component letters (A, B)
    are preserved, so binary components do not merge."""
    if s is None:
        return ''
    t = ' '.join(str(s).split())
    if t in ('', '--', 'nan', 'None', '0'):
        return ''
    if t.startswith('* '):
        t = t[2:]
    t = _BAYER_PREFIX.sub(lambda m: m.group(1), t)
    return t.casefold()
